# Agente de IA Minimalista - versão Gemini (v1: portabilidade)

Esta versão é o notebook do professor **portado do OpenAI (Responses API) para o Google Gemini (API nativa)**,
usando o nível gratuito da API. O cenário do incidente e as 7 ferramentas continuam **idênticos ao original**.
O que mudou nesta etapa:

- Modelo: `gpt-5.6-luna` -> `gemini-3.1-flash-lite`
- Formato das mensagens e das tools (o Gemini usa `functionDeclarations`, `functionCall` e `functionResponse`)
- Sem biblioteca externa: só Python padrão (`urllib`), nada para instalar
- Nova proteção: nova tentativa automática quando a API limita a taxa (erro 429/503)

In [ ]:
import json
import os
import time
import urllib.error
import urllib.request
from copy import deepcopy
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ").strip()

BASE_URL = "https://generativelanguage.googleapis.com/v1beta"
MODEL = "gemini-3.1-flash-lite"   # troque aqui para testar outro modelo Flash liberado na sua conta
MAX_RODADAS = 8
MAX_OUTPUT_TOKENS = 1200          # no Gemini, os tokens de "raciocinio" tambem contam neste limite
THINKING_LEVEL = "low"            # nivel de raciocinio (minimal / low / medium / high)

# O plano gratuito nao cobra. Os precos abaixo (GPT-5.6 Luna, do notebook original) servem
# apenas como referencia para calcular o "custo equivalente" se isto rodasse em API paga.
LIMITE_ESTIMADO_USD = 0.02
PRECO_INPUT_1M = 0.20
PRECO_OUTPUT_1M = 1.20

## Cenário original (sem alterações)

In [ ]:
CENARIO_ORIGINAL = {
    "metricas": {
        "api-gateway": {
            "cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200,
            "taxa_erro_pct": 18.4, "instancias": 3
        },
        "recommendation-api": {
            "cpu_pct": 98, "memoria_pct": 84, "latencia_p95_ms": 5870,
            "taxa_erro_pct": 31.7, "instancias": 48
        },
        "catalog-api": {
            "cpu_pct": 42, "memoria_pct": 55, "latencia_p95_ms": 93,
            "taxa_erro_pct": 0.2, "instancias": 4
        },
        "checkout-api": {
            "cpu_pct": 47, "memoria_pct": 58, "latencia_p95_ms": 310,
            "taxa_erro_pct": 0.6, "instancias": 5
        },
    },
    "logs": {
        "api-gateway": [
            "14:03:11 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:14 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:18 ERROR circuit breaker recommendation-api OPEN",
        ],
        "recommendation-api": [
            "14:02:57 WARN catalog response missing field related_items",
            "14:02:57 INFO invoking fallback target=recommendation-api depth=1",
            "14:02:58 INFO invoking fallback target=recommendation-api depth=8",
            "14:02:59 ERROR maximum fallback depth reached depth=25",
            "14:03:00 WARN request queue saturation=96%",
        ],
        "catalog-api": [
            "14:02:55 INFO GET /items/842 200 71ms",
            "14:03:01 INFO GET /items/163 200 66ms",
        ],
        "checkout-api": [
            "14:03:02 INFO POST /checkout 200 288ms",
            "14:03:07 INFO POST /checkout 200 302ms",
        ],
    },
    "deploys": {
        "api-gateway": [{"id": "dep-410", "versao": "5.8.1", "hora": "ontem 16:20", "status": "concluido"}],
        "recommendation-api": [
            {
                "id": "dep-442", "versao": "2.4.0", "hora": "13:58", "status": "concluido",
                "mudancas": [
                    "novo fallback para resposta incompleta do catalogo",
                    "FALLBACK_SERVICE: catalog-api -> recommendation-api",
                ],
            },
            {"id": "dep-431", "versao": "2.3.7", "hora": "ontem 11:40", "status": "concluido"},
        ],
        "catalog-api": [{"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"}],
        "checkout-api": [{"id": "dep-398", "versao": "7.2.0", "hora": "5 dias atras", "status": "concluido"}],
    },
    "configuracoes": {
        "api-gateway": {"timeout_ms": 5000, "retries": 1},
        "recommendation-api": {
            "fallback_service": "recommendation-api",
            "fallback_max_depth": 25,
            "autoscaling_min": 2,
            "autoscaling_max": 50,
            "autoscaling_cpu_target_pct": 60,
        },
        "catalog-api": {"cache_ttl_s": 300, "replicas": 4},
        "checkout-api": {"replicas": 5, "timeout_ms": 1500},
    },
    "custos": {
        "api-gateway": {"baseline_usd_h": 0.22, "atual_usd_h": 0.24},
        "recommendation-api": {"baseline_usd_h": 0.31, "atual_usd_h": 7.44},
        "catalog-api": {"baseline_usd_h": 0.38, "atual_usd_h": 0.40},
        "checkout-api": {"baseline_usd_h": 0.51, "atual_usd_h": 0.52},
    },
    "eventos": [
        {"hora": "13:58", "tipo": "deploy", "servico": "recommendation-api", "id": "dep-442"},
        {"hora": "14:01", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "2 -> 18 instancias"},
        {"hora": "14:03", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "18 -> 48 instancias"},
        {"hora": "14:03", "tipo": "alerta", "servico": "api-gateway", "detalhe": "latencia p95 acima de 5s"},
    ],
}

ambiente = deepcopy(CENARIO_ORIGINAL)
print("Serviços monitorados:", ", ".join(ambiente["metricas"].keys()))

## Ferramentas (funções Python, sem alterações)

In [ ]:
def _validar_servico(servico):
    if servico not in ambiente["metricas"]:
        return {"erro": f"Servico desconhecido: {servico}", "disponiveis": list(ambiente["metricas"])}
    return None


def consultar_metricas(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, **ambiente["metricas"][servico]}


def buscar_logs(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "linhas": ambiente["logs"].get(servico, [])}


def consultar_deploys(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "deploys": ambiente["deploys"].get(servico, [])}


def consultar_configuracao(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "configuracao": ambiente["configuracoes"].get(servico, {})}


def consultar_custos(servico):
    erro = _validar_servico(servico)
    if erro:
        return erro
    custo = ambiente["custos"][servico]
    multiplicador = round(custo["atual_usd_h"] / custo["baseline_usd_h"], 1)
    return {"servico": servico, **custo, "multiplicador": multiplicador}


def consultar_eventos_recentes():
    return {"eventos": ambiente["eventos"]}


def solicitar_rollback(deploy_id, justificativa):
    # Esta ferramenta NAO executa o rollback. Ela abre uma solicitacao.
    encontrado = None
    for servico, deploys in ambiente["deploys"].items():
        for deploy in deploys:
            if deploy["id"] == deploy_id:
                encontrado = {"servico": servico, **deploy}
                break
    if not encontrado:
        return {"status": "rejeitado", "motivo": "deploy inexistente"}
    return {
        "status": "AGUARDANDO_APROVACAO_HUMANA",
        "deploy": encontrado,
        "justificativa_registrada": justificativa,
        "executado": False,
    }

In [ ]:
print(json.dumps(consultar_metricas("api-gateway"), indent=2, ensure_ascii=False))
print(json.dumps(consultar_custos("recommendation-api"), indent=2, ensure_ascii=False))

## Declaração das tools no formato Gemini

Diferenças em relação ao formato OpenAI: as funções ficam dentro de `functionDeclarations`,
os tipos são em maiúsculas (`OBJECT`, `STRING`) e não existem `strict` nem `additionalProperties`.

In [ ]:
SERVICOS = ["api-gateway", "recommendation-api", "catalog-api", "checkout-api"]


def decl_servico(nome, descricao):
    return {
        "name": nome,
        "description": descricao,
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "servico": {"type": "STRING", "enum": SERVICOS, "description": "Nome do servico"}
            },
            "required": ["servico"],
        },
    }


DECLARACOES = [
    decl_servico("consultar_metricas", "Consulta CPU, memoria, latencia, erros e numero de instancias de um servico."),
    decl_servico("buscar_logs", "Busca as linhas recentes de log de um servico. Use para investigar sintomas e erros."),
    decl_servico("consultar_deploys", "Lista deploys recentes e mudancas conhecidas de um servico."),
    decl_servico("consultar_configuracao", "Consulta a configuracao operacional atual de um servico."),
    decl_servico("consultar_custos", "Compara o custo atual por hora de um servico com seu baseline."),
    {
        "name": "consultar_eventos_recentes",
        "description": "Lista eventos cloud recentes em ordem temporal: deploys, autoscaling e alertas.",
    },
    {
        "name": "solicitar_rollback",
        "description": "Solicita rollback de um deploy suspeito. Apenas abre pedido; exige aprovacao humana para executar.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "deploy_id": {"type": "STRING", "description": "ID exato de um deploy consultado"},
                "justificativa": {"type": "STRING", "description": "Evidencia objetiva que justifica o rollback"},
            },
            "required": ["deploy_id", "justificativa"],
        },
    },
]

TOOLS = [{"functionDeclarations": DECLARACOES}]
print(f"{len(DECLARACOES)} ferramentas declaradas para o modelo.")

In [ ]:
# dispatcher
FUNCOES_PERMITIDAS = {
    "consultar_metricas": consultar_metricas,
    "buscar_logs": buscar_logs,
    "consultar_deploys": consultar_deploys,
    "consultar_configuracao": consultar_configuracao,
    "consultar_custos": consultar_custos,
    "consultar_eventos_recentes": consultar_eventos_recentes,
    "solicitar_rollback": solicitar_rollback,
}


def executar_ferramenta(nome, argumentos):
    if nome not in FUNCOES_PERMITIDAS:
        return {"erro": f"Ferramenta nao permitida: {nome}"}
    try:
        return FUNCOES_PERMITIDAS[nome](**argumentos)
    except Exception as exc:
        return {"erro": type(exc).__name__, "detalhe": str(exc)}

In [ ]:
INSTRUCOES = '''
Voce e o agente legista de incidentes Cloud da Loja Nebulosa.
Seu objetivo e investigar a causa raiz usando apenas evidencias retornadas pelas ferramentas.

Regras:
- Nao trate proximidade temporal como prova de causalidade.
- Investigue primeiro; nao chute valores ou eventos.
- Faca consultas curtas e seletivas.
- Diferencie sintoma, causa raiz e impacto financeiro.
- Se houver evidencia suficiente, solicite rollback com justificativa objetiva.
- Nunca diga que uma acao foi executada quando a ferramenta disser que aguarda aprovacao.
- Encerre com: diagnostico, evidencias, impacto, acao recomendada e nivel de confianca.
- Seja conciso.
'''.strip()

INCIDENTE = '''
INC-2026-0911: desde 14h03, usuarios relatam lentidao no e-commerce.
O api-gateway disparou alerta de latencia e o custo cloud esta subindo.
Descubra a causa raiz e recomende a resposta operacional mais segura.
'''.strip()

## Chamada à API Gemini e laço do agente

Pontos importantes desta adaptação:
- O conteúdo devolvido pelo modelo é guardado **sem alteração** no histórico. Os modelos Gemini 3 incluem uma
  `thoughtSignature` que precisa voltar na rodada seguinte; copiar o conteúdo inteiro garante isso.
- Os tokens de raciocínio (`thoughtsTokenCount`) são somados aos de saída.
- Erros 429/500/503 disparam nova tentativa automática com espera crescente.

In [ ]:
def chamar_gemini(corpo, tentativas=4):
    url = f"{BASE_URL}/models/{MODEL}:generateContent"
    for tentativa in range(1, tentativas + 1):
        req = urllib.request.Request(
            url,
            data=json.dumps(corpo).encode("utf-8"),
            method="POST",
            headers={"Content-Type": "application/json", "X-goog-api-key": os.environ["GEMINI_API_KEY"]},
        )
        try:
            with urllib.request.urlopen(req, timeout=90) as resposta:
                return json.loads(resposta.read())
        except urllib.error.HTTPError as erro:
            detalhe = erro.read().decode("utf-8", errors="replace")[:500]
            config = corpo.get("generationConfig", {})
            if erro.code == 400 and "thinking" in detalhe.lower() and "thinkingConfig" in config:
                print("  (modelo nao aceita thinkingConfig; repetindo sem ele)")
                del config["thinkingConfig"]
                continue
            if erro.code in (429, 500, 503) and tentativa < tentativas:
                espera = 15 * tentativa
                print(f"  (API respondeu {erro.code}; aguardando {espera}s para tentar de novo...)")
                time.sleep(espera)
                continue
            raise RuntimeError(f"Erro {erro.code} da API Gemini: {detalhe}") from None


def estimar_custo(input_tokens, output_tokens):
    return (
        input_tokens * PRECO_INPUT_1M / 1_000_000
        + output_tokens * PRECO_OUTPUT_1M / 1_000_000
    )


def investigar_incidente(incidente=INCIDENTE):
    historico = [{"role": "user", "parts": [{"text": incidente}]}]
    total_input = 0
    total_output = 0
    ferramentas_usadas = []

    for rodada in range(1, MAX_RODADAS + 1):
        corpo = {
            "systemInstruction": {"parts": [{"text": INSTRUCOES}]},
            "contents": historico,
            "tools": TOOLS,
            "toolConfig": {"functionCallingConfig": {"mode": "AUTO"}},
            "generationConfig": {
                "maxOutputTokens": MAX_OUTPUT_TOKENS,
                "thinkingConfig": {"thinkingLevel": THINKING_LEVEL},
            },
        }
        resposta = chamar_gemini(corpo)

        uso = resposta.get("usageMetadata", {})
        total_input += uso.get("promptTokenCount", 0)
        total_output += uso.get("candidatesTokenCount", 0) + uso.get("thoughtsTokenCount", 0)
        custo = estimar_custo(total_input, total_output)
        print(f"\n--- Rodada {rodada} | custo equivalente acumulado: US$ {custo:.6f} (real: US$ 0, plano gratuito) ---")

        candidatos = resposta.get("candidates") or []
        if not candidatos:
            print("A API nao devolveu candidatos:", json.dumps(resposta.get("promptFeedback", {}), ensure_ascii=False))
            return {"resposta": None, "interrompido": True, "rodadas": rodada,
                    "input_tokens": total_input, "output_tokens": total_output,
                    "custo_equivalente_usd": custo, "ferramentas": ferramentas_usadas}

        conteudo = candidatos[0].get("content", {"role": "model", "parts": []})
        partes = conteudo.get("parts", [])
        historico.append(conteudo)  # preserva a thoughtSignature
        chamadas = [p["functionCall"] for p in partes if "functionCall" in p]

        if not chamadas:
            texto = "".join(p.get("text", "") for p in partes if not p.get("thought"))
            if candidatos[0].get("finishReason") == "MAX_TOKENS":
                print("AVISO: resposta cortada por MAX_OUTPUT_TOKENS; aumente o limite.")
            print("\nRELATORIO FINAL\n")
            print(texto)
            return {
                "resposta": texto,
                "input_tokens": total_input,
                "output_tokens": total_output,
                "custo_equivalente_usd": custo,
                "rodadas": rodada,
                "ferramentas": ferramentas_usadas,
            }

        respostas = []
        for chamada in chamadas:
            nome = chamada["name"]
            argumentos = chamada.get("args", {})
            print(f"Agente pediu: {nome}({argumentos})")
            resultado = executar_ferramenta(nome, argumentos)
            print("Observacao:", json.dumps(resultado, ensure_ascii=False))
            ferramentas_usadas.append(nome)
            retorno = {"name": nome, "response": {"result": resultado}}
            if "id" in chamada:
                retorno["id"] = chamada["id"]
            respostas.append({"functionResponse": retorno})
        historico.append({"role": "user", "parts": respostas})

        if custo >= LIMITE_ESTIMADO_USD:
            print(f"\nLimite didatico de US$ {LIMITE_ESTIMADO_USD:.2f} (equivalente) atingido.")
            return {"resposta": None, "interrompido": True, "rodadas": rodada,
                    "input_tokens": total_input, "output_tokens": total_output,
                    "custo_equivalente_usd": custo, "ferramentas": ferramentas_usadas}

    print("\nNumero maximo de rodadas atingido. O agente foi interrompido pelo software.")
    return {"resposta": None, "interrompido": True, "rodadas": MAX_RODADAS,
            "input_tokens": total_input, "output_tokens": total_output,
            "custo_equivalente_usd": estimar_custo(total_input, total_output),
            "ferramentas": ferramentas_usadas}

## Execução do cenário original (Print 1)

In [ ]:
resultado = investigar_incidente()
resultado

## Próximas etapas (roteiro do professor)

1. Incluir uma tool temporal no agente
   1.1. Modificar o mockup de métricas: criar um problema novo que exige investigação temporal
2. Testar e avaliar se o agente funciona bem
3. Portar para uma nova LLM (feito nesta versão: OpenAI -> Gemini)